## Extract CPT Data from CallHome Chat files and Perform inspection

First load the file.

In [1]:
import pylangacq

chats = pylangacq.read_chat("../spa.zip")
print(chats.head())

*A:    sí         ,          eso              es                       para      eso              ,          de          seguro            .
%mor:  intj|sí    cm|cm      pron|ese-Dem-S1  aux|ser-Fin-Ind-Pres-S3  adp|para  pron|ese-Dem-S1  cm|cm      adp|de      noun|seguro-Masc  .
%gra:  1|4|ADVCL  2|1|PUNCT  3|4|NSUBJ        4|0|ROOT                 5|6|CASE  6|4|OBL          7|8|PUNCT  8|6|ADVMOD  9|8|FIXED         10|4|PUNCT
       ⏱ 123384–124304 ms

*A:    no          importa                        .
%mor:  adv|no      verb|importar-Fin-Ind-Pres-S3  .
%gra:  1|2|ADVMOD  2|0|ROOT                       3|2|PUNCT

*B:    no          importa                        .
%mor:  adv|no      verb|importar-Fin-Ind-Pres-S3  .
%gra:  1|2|ADVMOD  2|0|ROOT                       3|2|PUNCT
       ⏱ 123985–124525 ms

*B:    bueno        aquí        ,          la                       Zaida        está                       estudiando           también       en         la                       univer

Now extract files and speaker exchanges into a multi-dimensional list object.

In [2]:
chats_sorted = chats.words(by_utterance=True, by_file=True)

print(chats_sorted[0][:5])

[['sí', ',', 'eso', 'es', 'para', 'eso', ',', 'de', 'seguro', '.'], ['no', 'importa', '.'], ['no', 'importa', '.'], ['bueno', 'aquí', ',', 'la', 'Zaida', 'está', 'estudiando', 'también', 'en', 'la', 'universidad', 'con', 'la', 'Liana', '.'], ['y', 'qué', 'estudia', ',', 'mama', ',', 'qué', 'están', 'estudiando', '.']]


Check that each file is it's own list object (should get 140 as there are 140 chat files in the corpus)

In [3]:
print(len(chats_sorted))

140


Now we can extract the data to a dataframe for further inspection

In [4]:
# import re

# # Optimized Regex patterns with explicit capture groups for data retention
# regex_patterns = {
#     "square_bracket_pattern": re.compile(r'(\[\[[^\]]+\]\])|\s?\[[^\]]+\]'),
#     "unintelligible_pattern": re.compile(r'\s?\(\(([^)]*)\)\)'),
#     "element_tags_pattern": re.compile(r'\s?<([^>]+)>'),
#     "simultaneous_speech_pattern": re.compile(r'\s?#([^#]+)#'),
#     "aside_pattern": re.compile(r'\s?//[^/]*//'),
#     "mispronunciation_pattern": re.compile(r'\s?\+([^+]+)\+'),
#     "incomplete_mispronunciation_pattern": re.compile(r'\s?\+(\.\.\.)'),
#     "idiosyncratic_pattern": re.compile(r'\s?\*\*([^*]+)\*\*'),
#     "cut_off_pattern": re.compile(r' ?-- ?'),
# }

# def _process_element_tag(match: re.Match) -> str:
#     """Helper to process the routing logic inside angle brackets."""
#     inner_text = match.group(1)
    
#     if "((" in inner_text or "))" in inner_text:
#         return ""
    
#     if "?" in inner_text:
#         # Strip the question mark and format cleanly
#         return " " + inner_text.replace("?", "").strip()
    
#     underscore_index = inner_text.find("_")
#     if underscore_index != -1:
#         return " " + inner_text[underscore_index + 1:].replace("_", " ")
        
#     return " " + inner_text

# def regex_pipeline(text: str) -> str:
#     """Apply regex patterns to clean the text via single-pass lambda substitution."""
    
#     # 1. Square brackets: Keep VIP [[ ]], drop [ ]
#     text = regex_patterns["square_bracket_pattern"].sub(lambda m: m.group(1) if m.group(1) else "", text)

#     # 2. Unintelligible: Drop if empty, keep inner text if populated
#     text = regex_patterns["unintelligible_pattern"].sub(
#         lambda m: " " + m.group(1).strip() if m.group(1).strip() else "", text
#     )

#     # 3. Element tags: Apply complex routing logic via callable
#     text = regex_patterns["element_tags_pattern"].sub(_process_element_tag, text)

#     # 4. Simultaneous speech: Extract inner text
#     text = regex_patterns["simultaneous_speech_pattern"].sub(r' \1', text)

#     # 5. Asides: Destroy entirely
#     text = regex_patterns["aside_pattern"].sub("", text)

#     # 6. Mispronunciation: Extract inner text
#     text = regex_patterns["mispronunciation_pattern"].sub(r' \1', text)

#     # 6.5. Incomplete Mispronunciation: Extract trailing ellipses for unclosed tags
#     text = regex_patterns["incomplete_mispronunciation_pattern"].sub(r' \1', text)

#     # 7. Idiosyncratic words: Extract inner text
#     text = regex_patterns["idiosyncratic_pattern"].sub(r' \1', text)

#     # 8. Cut-offs: Replace with single hyphen
#     text = regex_patterns["cut_off_pattern"].sub("-", text)

#     # 9. Format normalization: Collapse orphaned double spaces caused by internal tag extraction
#     text = re.sub(r' {2,}', ' ', text)

#     return text.strip()

In [5]:
from typing import List

def clean_utterance(utterance: List[str]) -> str:
    """
    Clean the utterance by removing any non-word tokens and converting to lowercase.
    """
    words_str = ""
    for word in utterance:
        if len(word) <= 1:
            words_str += word
        else:
            words_str += " " + word
    
    # cleaned_text = regex_pipeline(words_str)
    return words_str.strip()

In [6]:
import pandas as pd

master_df = pd.DataFrame(columns=["file_path", "utterance_num", "utterance", "speaker"])

# You can iterate over chat.utterances() which gives you a flat list 
# of Utterance objects for the entire dataset, or by_file=True if you 
# want to keep files separated.
for chat_file_path, file_utterances in zip(chats.file_paths, chats.utterances(by_file=True)):
    
    raw_words = []
    utterance_words = []
    speaker_map = []

    temp_df = pd.DataFrame(columns=["file_path", "utterance_num", "utterance", "speaker"])
    
    # Iterate through the actual Utterance objects
    for u in file_utterances:
        # 2. Extract the clean words from the utterance's tokens
        words = [token.word for token in u.tokens]
        # print(words)

        # raw_words_str = ""
        # for word in words:
        #     if len(word) <= 1:
        #         raw_words_str += word
        #     else:
        #         raw_words_str += " " + word

        # raw_words.append(raw_words_str.strip())        

        words_str = clean_utterance(words)
        
        # check if this is an orphaned grammatical marker (e.g., a single punctuation mark) and skip it if so
        if len(words_str.strip()) <= 1:
            continue

        cleaned_utterance = clean_utterance(words_str.strip())

        utterance_words.append(cleaned_utterance)
        # Get the speaker for this specific utterance (e.g., 'CHI', 'MOT')
        speaker_map.append(u.participant)

    # cleaned_utterance = clean_utterance(utterance_words)

    # print(f"File: {chat_file_path}")
    # print("Raw Words:", raw_words)
    # print("Words:", utterance_words)
    # print("Speakers:", speaker_map)
    # print("Cleaned Utterance:", cleaned_utterance)

    temp_df = pd.DataFrame({
        "file_path": [chat_file_path] * len(utterance_words),
        "utterance_num": list(range(len(utterance_words))),
        "utterance": utterance_words,
        "speaker": speaker_map
    })

    # print(temp_df)

    master_df = pd.concat([master_df, temp_df], ignore_index=True)

# create unique utterance IDs
master_df.reset_index(names='utterance_id', inplace=True)

print(master_df.describe())


       utterance_id
count  31198.000000
mean   15598.500000
std     9006.231185
min        0.000000
25%     7799.250000
50%    15598.500000
75%    23397.750000
max    31197.000000


In [7]:
import torch
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset
from collections import Counter
import datasets
from tqdm import tqdm

# 1. Initialize the model on the GPU
# device="cuda" (or device=0) forces the model into VRAM
classifier = pipeline(
    "ner", 
    model="sagorsarker/codeswitch-spaeng-lid-lince", 
    aggregation_strategy="simple",
    device="mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else -1)
)

def extract_cpt_candidates_batched(text_corpus: pd.DataFrame, batch_size: int = 4096) -> list[tuple]:
    """
    Executes batched token-classification on a high-VRAM GPU.
    """
    en_candidate_counter = Counter()
    other_candidate_counter = Counter()

    print("Converting utterances to dataset for batched inference...")
    text_dataset = datasets.Dataset.from_pandas(text_corpus, split="train")
    
    # slice for only first 100 utterances
    # text_dataset = text_dataset.select(range(0, 100))

    pipeline_iterator = classifier(
        KeyDataset(text_dataset, "utterance"),
        batch_size=batch_size
    )
    # print(f"Text dataset: {text_dataset}")
    # print(f"First 5 utterances: {text_dataset[:5]}")
    
    # 3. The pipeline returns a list of lists (one list of entities per utterance)
    for utterance_entities in tqdm(pipeline_iterator, total=len(text_dataset)):
        for entity in utterance_entities:
            # print(f"Entity: {entity}")
            # if entity['entity_group'] != 'spa' and entity['entity_group'] != 'other':
            #     print(f"Entity: {entity}")
            if entity['entity_group'] == 'en':
                
                # Clean up subword hashes from the tokenizer
                word = entity['word'].replace('##', '').strip().lower()
                
                if len(word) > 2:
                    en_candidate_counter[word] += 1
            
            elif entity['entity_group'] == 'other':
                word = entity['word'].replace('##', '').strip().lower()
                
                if len(word) > 2:
                    other_candidate_counter[word] += 1
        
        # break

    # nothing super interesting after 50                
    return (en_candidate_counter.most_common(), other_candidate_counter.most_common())

# --- Execution ---
def execute_langid_analysis(dataframe: pd.DataFrame) -> None:
    results = extract_cpt_candidates_batched(dataframe)

    print("English Candidates:")
    for candidate, count in results[0][:50]:  # Access the first tuple (English candidates)
        print(f"{candidate}: {count}")

    print("-"*80)
    print("-"*80)
    print("Other Candidates:")
    for candidate, count in results[1][:50]:  # Access the second tuple (Other candidates)
        print(f"{candidate}: {count}")

execute_langid_analysis(master_df)

/Users/dzurec/ai-projects/es-callhome-cleaner/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20144.00it/s]
[transformers] BertForTokenClassification LOAD REPORT from: sagorsarker/codeswitch-spaeng-lid-lince
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Converting utterances to dataset for batched inference...


100%|██████████| 31198/31198 [00:45<00:00, 679.93it/s] 

English Candidates:
noise: 225
has: 146
crying: 34
clear: 30
vocalization: 21
oye: 20
once: 15
dime: 11
fax: 10
thanksgiving: 9
day: 8
hom: 6
e - mail: 6
american: 6
master: 6
mmm: 6
modem: 6
college: 6
kung fu: 6
o sea: 5
lay: 5
laugh: 5
jet: 5
very: 4
oyey: 4
late: 4
green: 4
t - shirt: 4
' s: 4
postaly: 4
restaurant: 4
red: 4
net: 4
full: 4
email: 4
avenue: 4
halloween: 4
shack: 4
heavy: 4
western: 4
background noise: 3
mail: 3
card: 3
lunch: 3
mmm noise: 3
pld: 3
hello: 3
roommates: 3
bye: 3
tickets: 3
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
Other Candidates:
+...: 3058
+ /.: 510
, +...: 16
y +...: 7
☺no,: 2
+ / /.: 2
, o +...: 1
b :: 1
oo?: 1
: :.: 1
g -: 1
o : :: 1
cuento☺.: 1
il +...: 1
bautizo☺.: 1
☺entonces: 1
niño☺.: 1
pera☺.: 1
vez☺.: 1
☺dice: 1
otro☺.: 1
☺cómo: 1
flaco☺.: 1
, ☺que: 1
cortita☺.: 1
☺las guaguas☺!: 1
c -.: 1
novio☺.: 1
, ☺mira: 1
casualidad

Okay, interesting results... Let's look at the data a little more. Let's see what the actual utterance containing some of these "English Canidates" look like. We can sample 10 random examples for each "English" token the LID model identified where we know there is more than 10 in the dataset. Anything less isn't worth our time.

In [8]:
# adjust pandas settings to display the true sample without truncation
pd.set_option('display.max_colwidth', None)

# adjust
hit_list = ["noise", "has", "crying", "oye", "clear", "vocalization", "once", "dime", "_", r"\.\.\.", r"\+\s?/\.", r",\.\.\."]

utterances_shuffled = master_df.sample(frac=1, random_state=42).reset_index(drop=True)

# search for hits

for word in hit_list:
    print(f"Searching for '{word}' in utterances...")
    hits = utterances_shuffled[utterances_shuffled['utterance'].str.contains(word, case=False, na=False)]
    print(f"Found {len(hits)} hits for '{word}':")
    print(hits[['file_path', 'utterance', 'speaker']][:10])  # Show only the first 10 hits for brevity
    print("-"*80)

Searching for 'noise' in utterances...
Found 240 hits for 'noise':
    file_path                                              utterance speaker
141  1198.cha                                                 noise.      B1
326  1198.cha  sí, sí, todito,y lo han mandadoa hacer biopsia noise.      B1
362  1198.cha                                                 noise.      B1
412  0950.cha                                      noise ya, ya veo.       A
460  1411.cha                                           bueno noise.       B
501  1198.cha                                                 noise.      B1
624  0291.cha                                                 noise.       B
644  1122.cha        estaba muy emocionada al principio, pero noise.       B
672  1807.cha                              Mira, para acá noise +...       B
780  1122.cha                                             ah, noise.       B
--------------------------------------------------------------------------------
Searc

### Discoveries About the Contamination

- `1198.cha` contains 200 hits for "noise" on ctrl + f inspection
- `has` is flagging on benign Spanish words like "hasta", "muchas", and "has" is a legitmate form of haber in reference to tu (you informal)
- `1255.cha` yields 52 hits for "crying" on ctrl + f inspection indicating it is a major source of the contamination for this word
- NOTE: leaked annotations like "noise" and "crying" appear to almost exclusively come at the beginning or end of an utterance
- `oye` is a conjugation of "oír" and legitmate Spanish
- `clear` is an artifact of pylangacq not parsing `&=throat clear` correctly, can be dropped in entirety without causing linguistic damage
- `vocalization` appears to be an artifact of parsing from annotations like `&=child vocalization` as demonstrated throughout `0731.cha`. Can confidently drop with regex filter
- NOTE: After removing the offending annotation artifact some lines may be completely blank, these turns should be completely omitted from the final corpus
- Keep `once` instances as these are either extracted from "entonces" or refer to the number "once" (11)
- Uses of `dime` seem to be legitimate and most of the time appears alone as "dime" (tell me)
- Ellipses `...` are a part of standard speech representing silence, the ellipses can stay in the corpus without modification
- `+/.` seems to represent almost "half baked" thoughts, we can remove them from the dataset with regex filter (meaning of text won't change)
- Additonal: instances of underscored English not being split correctly, for instance "British_Airways", 124 instances of English phrases seperated by 1 or more "_" across the dataset

#### Response Plan

- Keep upstream data "pure" until after this cell
- Ensure Removal Phase tracks the diff between original utterance and newly cleaned utterance to ensure proper Spanish is not being broken by our cleaning strategy (audit logging)
- Replace all instances of "background noise" in text with "", then replace all instances of "noise" with "" ONLY if it is surrounded by whitespace
- Replace all instances of "crying" in text with ""
- Replace all instances of "clear" with ""
- Replace all instances of "vocalization" with ""
- Rerun LID post cleaning and analyze results

In [9]:
import re

def replace_and_modify(row: pd.Series, pattern: str, replacement: str, cleaning_operation: str) -> pd.Series:

    # perform main cleaning operation
    row['utterance'] = re.sub(pattern, replacement, row['utterance'], flags=re.IGNORECASE)

    # track the modification
    row['utterance_modifications'].append(cleaning_operation)

    # return the modified row
    return row

UNDERSCORE_REGEX_PATTERN = re.compile(r'\S*_\S*')

def code_switching_cleaning(row: pd.Series) -> pd.Series:
    code_switches = re.findall(UNDERSCORE_REGEX_PATTERN, row['utterance'])

    for code_switch in code_switches:
        # Replace underscores with spaces
        cleaned_phrase = code_switch.replace('_', ' ')
        
        # Update the utterance
        row['utterance'] = row['utterance'].replace(code_switch, cleaned_phrase)
        
        # Track the modification
        row['utterance_modifications'].append(f"CODE SWITCHING: '{code_switch}' -> '{cleaned_phrase}'")

    return row

def update_master_df(master_df: pd.DataFrame, modified_rows: pd.DataFrame) -> pd.DataFrame:
    # set_index without inplace returns a new DataFrame with the updated index
    master_idx = master_df.set_index('utterance_id')
    modified_idx = modified_rows.set_index('utterance_id')

    # .update() operates in-place on the caller (master_idx)
    master_idx.update(modified_idx)

    # Reset the index to restore 'utterance_num' as a regular column
    return master_idx.reset_index()

def normalize_whitespace(df: pd.DataFrame, col: str = 'utterance',
                         note: str = "WHITESPACE NORMALIZED") -> pd.DataFrame:
    original = df[col]
    cleaned  = original.str.replace(r'\s+', ' ', regex=True).str.strip()
    changed  = original.notna() & cleaned.ne(original)

    df.loc[changed, col] = cleaned[changed]
    df.loc[changed, 'utterance_modifications'] = (
        df.loc[changed, 'utterance_modifications'].map(lambda mods: mods + [note])
    )
    return df

In [10]:
# keep 1198.cha for now even though it contains a lot of "noise" artifacts

# Also normalize English phrases seperated by "_" by casting to " "

# words to drop:
# words are surrounded by a regex pattern to ensure we do not create spacing issues (ie leaving double spaces)
# leave "background noise" as it needs special treatment
annotation_drop_filter = {
    "crying": r'\bcrying\b',
    "clear": r'\bclear\b',
    "vocalization": r'\bvocalization\b',
    "noise": r'\bnoise\b',
}

symbol_drop_filter = {
    "+...": (r'\s*\+\.\.\.', "..."),
    "+": (r'\+', "")
}

# track original utterances
original_utterances = master_df['utterance'].copy()
master_df['original_utterance'] = original_utterances

# track utterance modifications
master_df['utterance_modifications'] = [[] for _ in range(len(master_df))]

# first handle code switching
code_switching_rows = master_df[master_df['utterance'].str.contains('_', na=False)]

code_switching_rows = code_switching_rows.apply(code_switching_cleaning, axis=1)

# print(type(code_switching_rows))

# update master df
master_df = update_master_df(master_df, code_switching_rows)

# second filter and clean "background noise" rows
background_noise_rows = master_df[master_df['utterance'].str.contains("background noise", case=False, na=False)]

background_noise_rows = background_noise_rows.apply(lambda row: replace_and_modify(row, r"background noise", "", "ANNOTATION ARTIFACT REMOVED: 'background noise'"), axis=1)

# update master df
master_df = update_master_df(master_df, background_noise_rows)

# third filter out all other annotation artifacts rows only if artifact is surrounded by whitespace or at the start/end of the utterance (preserves these strings showing up in long Spanish words)

for artifact, pattern in annotation_drop_filter.items():
    artifact_rows = master_df[master_df['utterance'].str.contains(pattern, na=False)]
    
    artifact_rows = artifact_rows.apply(lambda row: replace_and_modify(row, pattern, " ", f"ANNOTATION ARTIFACT REMOVED: ' {artifact} '"), axis=1)
    
    # update master df
    master_df = update_master_df(master_df, artifact_rows)

# now take care of `+/.` artifacts first then run another clean for stray `+` that can be a part of `+...`
for artifact, (pattern, replacement) in symbol_drop_filter.items():
    artifact_rows = master_df[master_df['utterance'].str.contains(pattern, na=False)]
    
    artifact_rows = artifact_rows.apply(lambda row, p=pattern, r=replacement: 
                                        replace_and_modify(row, p, r, f"ANNOTATION SYMBOL NORMALIZED/REMOVED: ' {artifact} '"), 
                                        axis=1
                                    )
    
    # update master df
    master_df = update_master_df(master_df, artifact_rows)

# pass to clean up whitepace
# collapse spaces on all utterances
master_df = normalize_whitespace(master_df, col='utterance', note="WHITESPACE NORMALIZATION")

Run Langid over the new master df again

In [11]:
execute_langid_analysis(master_df)

Converting utterances to dataset for batched inference...


100%|██████████| 31198/31198 [00:46<00:00, 675.25it/s] 

English Candidates:
has: 146
oye: 33
once: 15
dime: 10
fax: 9
mmm: 8
thanksgiving: 8
hom: 6
e - mail: 6
american: 6
master: 6
modem: 6
college: 6
kung fu: 6
day: 5
o sea: 5
lay: 5
laugh: 5
jet: 5
free net: 5
very: 4
oyey: 4
late: 4
green card: 4
t - shirt: 4
' s: 4
postaly: 4
restaurant: 4
red: 4
email: 4
halloween: 4
heavy: 4
western union: 4
lunch: 3
pld: 3
hello: 3
roommates: 3
tickets: 3
martesy: 3
living: 3
show: 3
tod: 3
closet: 3
service: 3
tie: 3
offline: 3
quin: 3
state: 3
fahrenheit: 3
ovation: 3
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
Other Candidates:
...: 3062
,...: 16
y...: 4
☺no,: 2
/ /.: 2
xxx: 2
, o...: 1
b :: 1
oo?: 1
: :.: 1
g -: 1
o : :: 1
cuento☺.: 1
bautizo☺.: 1
☺entonces: 1
niño☺.: 1
pera☺.: 1
vez☺.: 1
☺dice: 1
otro☺.: 1
☺cómo: 1
flaco☺.: 1
, ☺que: 1
cortita☺.: 1
☺las guaguas☺!: 1
y /.: 1
c -.: 1
novio☺.: 1
, ☺mira: 1
casualidad☺.: 1
☺como dic

Looking much better! Let's inspect the rows that contain `green card` now to make sure this makes sense in context.

In [12]:
green_card_rows = master_df[master_df['utterance'].str.contains("green card", case=False, na=False)]
print(green_card_rows)

       utterance_id file_path utterance_num  \
2282           2282  0681.cha            92   
23423         23423  1759.cha           232   
23426         23426  1759.cha           235   
23428         23428  1759.cha           237   

                                                                                                                                utterance  \
2282                                      sí,y así le dices, sabes que yo estoy tratando de que me den la green card para poder trabajar.   
23423                                                                  yo estoy tratando, estoy haciendo los trámites de la green card...   
23426  el próximo año ya voya estar con la green card en un tiempo más, si es que me va todo bieny,y todo anda perfecto, yo voya tener...   
23428                                                                               la green card es la tempor- la residencia definitiva.   

      speaker  \
2282        B   
23423      A1   
23426  

Looks like it went well! On to the next step.

In [13]:
master_df.describe(include='all')

,utterance_id,file_path,utterance_num,utterance,speaker,original_utterance,utterance_modifications
count,31198.000000,31198,31198.0,31198,31198,31198,31198
unique,NaN,140,372.0,24191,8,24203,128
top,NaN,0713.cha,0.0,sí.,A,sí.,[]
freq,NaN,372,140.0,1003,14794,1003,27186
mean,15598.500000,NaN,NaN,NaN,NaN,NaN,NaN
std,9006.231185,NaN,NaN,NaN,NaN,NaN,NaN
min,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,7799.250000,NaN,NaN,NaN,NaN,NaN,NaN
50%,15598.500000,NaN,NaN,NaN,NaN,NaN,NaN
75%,23397.750000,NaN,NaN,NaN,NaN,NaN,NaN


Now drop any rows that might be empty after cleaning. For example before cleaning conversations in plain text could look like:

BEFORE:
```
A: noise
A: voy a la biblioteca
A: noise
B: Me encanta la biblioteca, la biblioteca es el mejor lugar para buscar buenos libros.
```

---

AFTER:
```
A: voy a la biblioteca
B: Me encatna la biblioteca, la biblioteca es el mejor lugar para buscar buenos libros.
```

Now we can reassemble the transcripts into clean documents for Continued Pretraining Ingestion!

In [16]:
def build_document_and_audit(subset: pd.Series) -> pd.Series:
    # 1. Sort the subset descending by utterance_num
    sorted_subset = subset.sort_values('utterance_num', ascending=True)
    
    # 2. Filter out NaN and empty strings
    valid_mask = (
        sorted_subset['utterance'].notna() & 
        (sorted_subset['utterance'].astype(str).str.strip() != "")
    )
    # Use .copy() to prevent SettingWithCopyWarning when we add the new column
    filtered = sorted_subset[valid_mask].copy() 
    
    # 3. RECALCULATE: Create a contiguous line number for the reconstructed doc
    filtered['doc_line_num'] = range(1, len(filtered) + 1)
    
    # 4. Reconstruct Document
    formatted_lines = (
        filtered['speaker'].astype(str) + 
        ": " + 
        filtered['utterance'].astype(str)
    )
    reconstructed_doc = '\n'.join(formatted_lines)
    
    # 5. Extract parallel lists
    cleaned_list = filtered['utterance'].tolist()
    original_list = filtered['original_utterance'].tolist()
    
    # 6. Build a Two-Way Audit Trail
    # Format: "Doc Line X (Raw ID Y) - Speaker: [modifications]"
    audit_trail = [
        f"Doc Line {doc_num} (Raw ID {int(raw_id)}) - {spk}: {mods}" 
        for doc_num, raw_id, spk, mods in zip(
            filtered['doc_line_num'],
            filtered['utterance_num'], 
            filtered['speaker'], 
            filtered['utterance_modifications']
        )
        if len(mods) > 0  # Only include entries with modifications
    ]
    
    return pd.Series({
        'reconstructed_document': reconstructed_doc,
        'cleaned_utterances': cleaned_list,
        'original_utterances': original_list,
        'audit_trail': audit_trail
    })

# Apply to each unique file_path
final_df = (
    master_df.groupby('file_path')
    .apply(build_document_and_audit)
    .reset_index() 
)

In [17]:
final_df.head(1)

file_path  \
0  0053.cha   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [18]:
final_df.describe(include='all')

file_path  \
count        140   
unique       140   
top     0053.cha   
freq           1   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

Inspect the audit trail of `0053.cha` specifically to ensure it makes sense and is parsable.

In [19]:
chat_0053 = final_df[final_df['file_path'] == "0053.cha"]

for entry in chat_0053['audit_trail'].iloc[0]:  # Access the first (and only) row's audit_trail
    print(entry)

Doc Line 34 (Raw ID 33) - B1: ["ANNOTATION ARTIFACT REMOVED: 'background noise'"]
Doc Line 36 (Raw ID 35) - B1: ["ANNOTATION SYMBOL NORMALIZED/REMOVED: ' +... '"]
Doc Line 46 (Raw ID 45) - A: ["ANNOTATION SYMBOL NORMALIZED/REMOVED: ' +... '"]
Doc Line 52 (Raw ID 51) - B1: ["ANNOTATION SYMBOL NORMALIZED/REMOVED: ' +... '"]
Doc Line 59 (Raw ID 58) - A: ["ANNOTATION SYMBOL NORMALIZED/REMOVED: ' +... '"]
Doc Line 65 (Raw ID 64) - A: ["ANNOTATION SYMBOL NORMALIZED/REMOVED: ' +... '"]
Doc Line 69 (Raw ID 68) - A: ["ANNOTATION SYMBOL NORMALIZED/REMOVED: ' +... '"]
Doc Line 71 (Raw ID 70) - A: ["ANNOTATION SYMBOL NORMALIZED/REMOVED: ' +... '"]
Doc Line 77 (Raw ID 76) - B1: ["ANNOTATION SYMBOL NORMALIZED/REMOVED: ' +... '"]
Doc Line 78 (Raw ID 77) - A: ["ANNOTATION SYMBOL NORMALIZED/REMOVED: ' +... '"]
Doc Line 79 (Raw ID 78) - B1: ["ANNOTATION SYMBOL NORMALIZED/REMOVED: ' +... '"]
Doc Line 87 (Raw ID 86) - B1: ["ANNOTATION SYMBOL NORMALIZED/REMOVED: ' +... '"]
Doc Line 105 (Raw ID 104) - A: ["

Excellent! Now that everything looks good to go we can export the final result to parquet!

In [20]:
final_df.to_parquet("callhome_es_cleaned_transcript_pretraining_docs.parquet")